# Knowledge Base Setup and Pre-Processing

## Load PDF

In [68]:
import pdfplumber

In [69]:
PDF_PATH = "..\data\Tokoku_Customer_Service_Manual.pdf"

# Load PDF and check text formats for section grouping
with pdfplumber.open(PDF_PATH) as pdf:
    print(f"Total pages: {len(pdf.pages)}\n")
    
    for page_num, page in enumerate(pdf.pages[:3], start=1):
        print(f"--- Page {page_num} ---")
        lines = page.extract_text_lines()
        
        for line in lines:
            if line["chars"]:
                font_size = line["chars"][0]["size"]
                font_name = line["chars"][0]["fontname"]
                text_preview = line["text"][:60]
                print(f"size: {font_size:.1f} | font: {font_name} | text: {text_preview}")
        print()

Total pages: 24

--- Page 1 ---
size: 14.0 | font: BAAAAA+Arial-BoldMT | text: TOKOKU
size: 10.0 | font: AAAAAA+ArialMT | text: | Customer Service & Usage Manual
size: 9.0 | font: AAAAAA+ArialMT | text: Version 1.0 | 2025
size: 36.0 | font: BAAAAA+Arial-BoldMT | text: TOKOKU
size: 14.0 | font: AAAAAA+ArialMT | text: Shop Easy, Live Smarter
size: 20.0 | font: BAAAAA+Arial-BoldMT | text: Customer Service, Usage Manual & FAQ
size: 12.0 | font: AAAAAA+ArialMT | text: Complete Platform Guide for Buyers and Sellers on Tokoku
size: 10.0 | font: BAAAAA+Arial-BoldMT | text: Version 1.0 Last Updated May 2025
size: 10.0 | font: AAAAAA+ArialMT | text: Operations & CS
size: 10.0 | font: BAAAAA+Arial-BoldMT | text: Prepared by Language English
size: 10.0 | font: AAAAAA+ArialMT | text: Department
size: 9.0 | font: AAAAAA+ArialMT | text: Tokoku Platform | Confidential Internal Document | Page 1 of

--- Page 2 ---
size: 14.0 | font: BAAAAA+Arial-BoldMT | text: TOKOKU
size: 10.0 | font: AAAAAA+ArialMT |

In [70]:
# Define heading text format
MAIN_HEADING_SIZE = 18    
SUB_HEADING_SIZE = 14     
BOLD_FONT_KEYWORD = "Bold"

# Exclude non-headings with similar text format
SKIP_TEXTS = [
    "TOKOKU",
    "| Customer Service & Usage Manual",
    "Version 1.0 | 2025",
    "Tokoku Platform | Confidential Internal Document",
    "SECTION"
]

# Define function to check sentence format
def get_line_type(line):
    if not line["chars"]:
        return "empty"
    size = round(line["chars"][0]["size"])
    font = line["chars"][0]["fontname"]
    is_bold = BOLD_FONT_KEYWORD in font
    
    if size >= MAIN_HEADING_SIZE and is_bold:
        return "main_heading"
    elif size >= SUB_HEADING_SIZE and is_bold:
        return "sub_heading"
    else:
        return "body"

# Define function to detect noise
def is_noise(line):
    text = line["text"].strip()
    if len(text) < 3:
        return True
    return any(text.startswith(skip) for skip in SKIP_TEXTS)

# two level structure
sections = []
current_section = None
current_subsection = {"heading": None, "text": "", "page": 1}

In [71]:
# Group text based on section and sub section headings
with pdfplumber.open(PDF_PATH) as pdf:
    for page_num, page in enumerate(pdf.pages[2:], start=3):
        lines = page.extract_text_lines()
        
        for line in lines:
            if not line["chars"]:
                continue
            if is_noise(line):
                continue
            
            line_type = get_line_type(line)
            
            if line_type == "main_heading":
                # save current subsection
                if current_subsection["text"].strip():
                    if current_section:
                        current_section["subsections"].append(current_subsection)
                # save current section
                if current_section:
                    sections.append(current_section)
                # start new section
                current_section = {
                    "heading": line["text"].strip(),
                    "page": page_num,
                    "subsections": []
                }
                current_subsection = {
                    "heading": "General",
                    "text": "",
                    "page": page_num
                }
                
            elif line_type == "sub_heading":
                # save current subsection before starting new one
                if current_subsection["text"].strip():
                    if current_section:
                        current_section["subsections"].append(current_subsection)
                # start new subsection
                current_subsection = {
                    "heading": line["text"].strip(),
                    "text": "",
                    "page": page_num
                }
                
            else:
                current_subsection["text"] += line["text"] + "\n"

    # save the last subsection and section
    if current_subsection["text"].strip() and current_section:
        current_section["subsections"].append(current_subsection)
    if current_section:
        sections.append(current_section)

# preview output
print(f"Total sections: {len(sections)}\n")
for section in sections[:3]:
    print(f"Section: {section['heading']}")
    print(f"  Subsections: {len(section['subsections'])}")
    for sub in section["subsections"]:
        word_count = len(sub["text"].split())
        print(f"    [{word_count:3d} words] {sub['heading']}")
    print()

Total sections: 12

Section: 1. Platform Overview
  Subsections: 4
    [ 38 words] General
    [ 29 words] 1.1 Vision & Mission
    [ 64 words] 1.2 Core Services
    [ 36 words] 1.3 Platform Statistics (2025)

Section: 2. Getting Started – Account & Registration
  Subsections: 3
    [101 words] 2.1 How to Register a Tokoku Account
    [ 51 words] 2.2 Account Verification
    [ 55 words] 2.3 Profile & Security Settings

Section: 3. Buyer Guide – How to Shop on Tokoku
  Subsections: 5
    [ 74 words] 3.1 Finding Products
    [ 65 words] 3.2 Identifying Trusted Products & Stores
    [ 99 words] 3.3 Purchase Process (Step-by-Step)
    [ 49 words] 3.4 Tracking Your Order
    [ 49 words] 3.5 Confirming Receipt of Goods



In [72]:
# Check section and subsection overview
total_subsections = 0

for i, section in enumerate(sections):
    section_words = sum(len(sub["text"].split()) for sub in section["subsections"])
    print(f"\nSection {i+1:02d} | Page {section['page']:02d} | {section_words:4d} words | {section['heading']}")
    
    for j, sub in enumerate(section["subsections"]):
        word_count = len(sub["text"].split())
        print(f"   Sub {j+1} | Page {sub['page']:02d} | {word_count:3d} words | {sub['heading']}")
        total_subsections += 1

print(f"\nTotal sections: {len(sections)}")
print(f"Total subsections: {total_subsections}")


Section 01 | Page 03 |  167 words | 1. Platform Overview
   Sub 1 | Page 03 |  38 words | General
   Sub 2 | Page 03 |  29 words | 1.1 Vision & Mission
   Sub 3 | Page 03 |  64 words | 1.2 Core Services
   Sub 4 | Page 03 |  36 words | 1.3 Platform Statistics (2025)

Section 02 | Page 05 |  207 words | 2. Getting Started – Account & Registration
   Sub 1 | Page 05 | 101 words | 2.1 How to Register a Tokoku Account
   Sub 2 | Page 05 |  51 words | 2.2 Account Verification
   Sub 3 | Page 05 |  55 words | 2.3 Profile & Security Settings

Section 03 | Page 06 |  336 words | 3. Buyer Guide – How to Shop on Tokoku
   Sub 1 | Page 06 |  74 words | 3.1 Finding Products
   Sub 2 | Page 06 |  65 words | 3.2 Identifying Trusted Products & Stores
   Sub 3 | Page 06 |  99 words | 3.3 Purchase Process (Step-by-Step)
   Sub 4 | Page 07 |  49 words | 3.4 Tracking Your Order
   Sub 5 | Page 07 |  49 words | 3.5 Confirming Receipt of Goods

Section 04 | Page 08 |  358 words | 4. Seller Guide – How to 

In [73]:
# Checking what's in the General subsection
print(sections[0]["subsections"][0]["text"])

Tokoku is a trusted e-commerce platform connecting millions of buyers and sellers across
Indonesia. Founded in 2018 and headquartered in Jakarta, Tokoku provides a safe, convenient,
and affordable online shopping ecosystem for all segments of the Indonesian population.



## Restructuring the Knowledge Base

In [74]:
import os
import json
from groq import Groq
from dotenv import load_dotenv
import re

In [75]:
# Load Groq API key from .env file
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), ".env"))

# Verify it loaded correctly
api_key = os.getenv("GROQ_API_KEY")
print("API key loaded:", "✅ Yes" if api_key else "❌ Not found")

API key loaded: ✅ Yes


In [76]:
# Initiate Groq API Client
client = Groq(api_key=os.getenv(api_key))

In [77]:
# Define valid knowledge base categories
CATEGORIES = ["FAQ", "Policy", "Troubleshooting", "Contact"]

In [78]:
EXTRACTION_PROMPT = """You are a knowledge base builder for a customer service chatbot.
Restructure the source text into a single knowledge base entry.

STRICT RULES:
1. Only use information explicitly stated in the source text
2. Do not add, infer, or invent any information not present in the source
3. The entry must be SELF-CONTAINED — a reader should understand it without needing other entries
4. Write information in complete sentences with full context
5. Include ALL details, steps, bullet points, and facts from the source text — nothing should be omitted
6. Return ONLY valid JSON — no preamble, no markdown, no explanation
7. Always return exactly ONE entry — never split into multiple entries

SOURCE SECTION: {section}
SOURCE SUBSECTION: {subsection}

SOURCE TEXT:
{text}

Return exactly ONE entry in this exact format:
{{
    "topic": "concise topic label describing what this subsection is about",
    "information": "complete restructured paragraph covering ALL information from the source text",
    "category": "one of: Policy, Troubleshooting, Contact",
    "tags": ["tag1", "tag2", "tag3"]
}}

WRITING GUIDELINES:
- Policy: state the rule AND its conditions/exceptions
- Troubleshooting: describe the problem AND include ALL solution steps in full
- Contact: include the channel AND when/how to use it
- For procedures: write all steps in order as a coherent narrative
- For numerical facts: always include their context
- Minimum 3 sentences"""

In [79]:
# Create a function to extract knowledge entries for non-faq section such as policy, troubleshooting, etc
def extract_kb_entries(section_heading, subsection_heading, text):
    prompt = EXTRACTION_PROMPT.format(
        section=section_heading,
        subsection=subsection_heading,
        text=text.strip()
    )
    
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1
    )
    
    raw = response.choices[0].message.content.strip()
    
    # Strip markdown fences
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    
    # Sanitize control characters
    raw = raw.strip()
    raw = re.sub(r'\t', ' ', raw)
    raw = re.sub(r' +', ' ', raw)
    
    entry = json.loads(raw)
    return entry  # single dict now, not a list

In [80]:
FAQ_EXTRACTION_PROMPT = """You are a knowledge base builder for a customer service chatbot.
Extract individual FAQ entries from the source text provided.

STRICT RULES:
1. Only use information explicitly stated in the source text
2. Do not add, infer, or invent any information not present in the source
3. Each entry must represent ONE distinct question and its complete answer
4. Write information in complete sentences with full context
5. Return ONLY valid JSON — no preamble, no markdown, no explanation
6. category field MUST always be exactly "FAQ" — never use any other value
7. topic field MUST be a short 2-5 word label — never copy the full question

SOURCE SECTION: {section}
SOURCE SUBSECTION: {subsection}

SOURCE TEXT:
{text}

Return a JSON array where each element is one FAQ pair:
[
    {{
        "topic": "2-5 word label only",
        "information": "complete answer including all relevant details, conditions, and steps",
        "category": "FAQ",
        "tags": ["tag1", "tag2", "tag3"]
    }}
]

WRITING GUIDELINES:
- Each entry = exactly one question-answer pair from the source
- Include the complete answer — never summarize or omit details
- If the answer has steps, include ALL steps in order
- Minimum 2 sentences per information field

EXAMPLE OF CORRECT OUTPUT:
{{
    "topic": "Password Reset Process",
    "information": "To reset your password, go to the login page and click Forgot Password. Enter your registered email address and click Send OTP. Enter the 6-digit OTP code sent to your email, then create a new password of minimum 8 characters.",
    "category": "FAQ",
    "tags": ["password", "reset", "login"]
}}"""

In [81]:
# Create a function to extract knowledge entries for the faq section
def extract_faq_entries(section_heading, subsection_heading, text):
    prompt = FAQ_EXTRACTION_PROMPT.format(
        section=section_heading,
        subsection=subsection_heading,
        text=text.strip()
    )
    
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1
    )
    
    raw = response.choices[0].message.content.strip()
    
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    
    raw = raw.strip()
    raw = re.sub(r'\t', ' ', raw)
    raw = re.sub(r' +', ' ', raw)
    
    entries = json.loads(raw)
    return entries  # list of dicts

In [82]:
all_entries = []
failed_subsections = []

for section in sections:
    for sub in section["subsections"]:
        if len(sub["text"].strip()) < 20:
            continue
        
        # Use different extraction strategy for FAQ subsections
        is_faq_subsection = "Frequently Asked Questions" in section["heading"]
        
        try:
            # For faq section
            if is_faq_subsection:
                entries = extract_faq_entries(
                    section_heading=section["heading"],
                    subsection_heading=sub["heading"],
                    text=sub["text"]
                )
            # For non-faq section
            else:
                entry = extract_kb_entries(
                    section_heading=section["heading"],
                    subsection_heading=sub["heading"],
                    text=sub["text"]
                )
                entries = [entry]
            
            for entry in entries:
                entry["section"] = section["heading"]
                entry["subsection"] = sub["heading"]
                entry["page_number"] = sub["page"]
                entry["source_file"] = PDF_PATH
                all_entries.append(entry)
            
            print(f"✅ {section['heading']} → {sub['heading']} | {len(entries)} entries extracted")
            
        except Exception as e:
            failed_subsections.append({
                "section": section["heading"],
                "subsection": sub["heading"],
                "error": str(e)
            })
            print(f"❌ {section['heading']} → {sub['heading']} | Error: {e}")

✅ 1. Platform Overview → General | 1 entries extracted
✅ 1. Platform Overview → 1.1 Vision & Mission | 1 entries extracted
✅ 1. Platform Overview → 1.2 Core Services | 1 entries extracted
✅ 1. Platform Overview → 1.3 Platform Statistics (2025) | 1 entries extracted
✅ 2. Getting Started – Account & Registration → 2.1 How to Register a Tokoku Account | 1 entries extracted
✅ 2. Getting Started – Account & Registration → 2.2 Account Verification | 1 entries extracted
✅ 2. Getting Started – Account & Registration → 2.3 Profile & Security Settings | 1 entries extracted
✅ 3. Buyer Guide – How to Shop on Tokoku → 3.1 Finding Products | 1 entries extracted
✅ 3. Buyer Guide – How to Shop on Tokoku → 3.2 Identifying Trusted Products & Stores | 1 entries extracted
✅ 3. Buyer Guide – How to Shop on Tokoku → 3.3 Purchase Process (Step-by-Step) | 1 entries extracted
✅ 3. Buyer Guide – How to Shop on Tokoku → 3.4 Tracking Your Order | 1 entries extracted
✅ 3. Buyer Guide – How to Shop on Tokoku → 3.5 

In [83]:
print(f"\nTotal entries extracted: {len(all_entries)}")
print(f"Failed subsections: {len(failed_subsections)}")


Total entries extracted: 62
Failed subsections: 0


In [84]:
import json

sample_indices = [0, 21, 51, 61]

for i in sample_indices:
    entry = all_entries[i]
    print(f"--- Entry {i+1} ---")
    print(f"Section    : {entry['section']}")
    print(f"Subsection : {entry['subsection']}")
    print(f"Category   : {entry['category']}")
    print(f"Tags       : {entry['tags']}")
    print(f"Topic      : {entry['topic']}")
    print(f"Information: {entry['information'][:200]}")
    print()

--- Entry 1 ---
Section    : 1. Platform Overview
Subsection : General
Category   : Policy
Tags       : ['Tokoku', 'E-commerce', 'Indonesia']
Topic      : Tokoku Overview
Information: Tokoku is a trusted e-commerce platform connecting millions of buyers and sellers across Indonesia. Founded in 2018 and headquartered in Jakarta, Tokoku provides a safe, convenient, and affordable onl

--- Entry 22 ---
Section    : 6. Shipping & Delivery
Subsection : 6.2 Special Delivery Services
Category   : Policy
Tags       : ['shipping', 'delivery', 'policy']
Topic      : Special Delivery Services
Information: Tokoku Express Same-Day is available for orders placed before 11:00 AM WIB and delivered on the same day in Jakarta, Surabaya, Bandung, Medan, and Makassar. Free Shipping is available for Tokoku Plus 

--- Entry 52 ---
Section    : 11. Frequently Asked Questions (FAQ)
Subsection : 11.4 Returns & Refunds
Category   : FAQ
Tags       : ['item', 'description', 'dispute']
Topic      : Item Descriptio

In [85]:
# Confirm information is complete, not truncated
for i in [0, 21, 51, 61]:
    entry = all_entries[i]
    word_count = len(entry["information"].split())
    last_char = entry["information"][-1]
    print(f"Entry {i+1} | {word_count} words | ends with: '{last_char}' | {entry['topic']}")

Entry 1 | 38 words | ends with: '.' | Tokoku Overview
Entry 22 | 89 words | ends with: '.' | Special Delivery Services
Entry 52 | 32 words | ends with: '.' | Item Description Dispute
Entry 62 | 80 words | ends with: '.' | Company Contact Information


In [86]:
invalid = [e for e in all_entries if e["category"] not in CATEGORIES]
print(f"Invalid categories: {len(invalid)}")
for e in invalid:
    print(f"  → {e['category']} | {e['topic']}")

Invalid categories: 0


## Hallucination Validation


In [87]:
def strip_markdown(text):
    text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text)
    text = re.sub(r'[*_`#]', '', text)
    return text.strip()

def validate_entry(entry, sections):
    # find matching source text
    source_text = ""
    for section in sections:
        if section["heading"] == entry["section"]:
            for sub in section["subsections"]:
                if sub["heading"] == entry["subsection"]:
                    source_text = sub["text"].lower()
                    break

    information = strip_markdown(entry["information"])

    # rule 1 — reject empty entries
    if len(information.strip()) == 0:
        return {"is_valid": False, "reason": "empty information"}

    # rule 2 — allow short atomic facts containing
    # numbers, emails, URLs, or codes
    has_number = any(c.isdigit() for c in information)
    has_email = "@" in information
    has_url = "." in information and len(information) > 5
    is_atomic = len(information.split()) <= 6

    if is_atomic and (has_number or has_email or has_url):
        return {"is_valid": True, "reason": "valid atomic fact"}

    # rule 3 — word match ratio against source text
    meaningful_tokens = [
        w.lower() for w in information.split() if len(w) > 4
    ]

    if not meaningful_tokens:
        topic_words = [w.lower() for w in entry["topic"].split() if len(w) > 4]
        if any(w in source_text for w in topic_words):
            return {"is_valid": True, "reason": "topic verified in source"}
        return {"is_valid": False, "reason": "no verifiable content"}

    matches = sum(1 for w in meaningful_tokens if w in source_text)
    match_ratio = matches / len(meaningful_tokens)

    return {
        "is_valid": match_ratio >= 0.4,
        "match_ratio": round(match_ratio, 2),
        "reason": "ok" if match_ratio >= 0.4 else "possible hallucination"
    }

In [88]:
# Run validation
valid_entries = []
flagged_entries = []

for i, entry in enumerate(all_entries):
    # Clean markdown from all entries regardless
    entry["information"] = strip_markdown(entry["information"])
    
    result = validate_entry(entry, sections)

    if result["is_valid"]:
        valid_entries.append(entry)
    else:
        flagged_entries.append({
            "index": i + 1,
            "section": entry["section"],
            "subsection": entry["subsection"],
            "topic": entry["topic"],
            "reason": result["reason"],
            "information": entry["information"]
        })

print(f"✅ Valid entries   : {len(valid_entries)}")
print(f"⚠️  Flagged entries : {len(flagged_entries)}")
print(f"📊 Validation rate : {len(valid_entries)/len(all_entries)*100:.1f}%")

if flagged_entries:
    print(f"\n--- Flagged Entries ---")
    for f in flagged_entries:
        print(f"\n[{f['index']}] {f['section']} → {f['subsection']}")
        print(f"  Topic  : {f['topic']}")
        print(f"  Reason : {f['reason']}")
        print(f"  Info   : {f['information'][:150]}")

✅ Valid entries   : 61
⚠️  Flagged entries : 1
📊 Validation rate : 98.4%

--- Flagged Entries ---

[9] 3. Buyer Guide – How to Shop on Tokoku → 3.2 Identifying Trusted Products & Stores
  Topic  : Identifying Trusted Products & Stores
  Reason : possible hallucination
  Info   : When shopping on Tokoku, look for certain indicators to ensure you're purchasing from a trusted store. These indicators include the presence of a Star


In [92]:
# Check entry 9's full information
print(all_entries[8]["information"])

When shopping on Tokoku, look for certain indicators to ensure you're purchasing from a trusted store. These indicators include the presence of a Star Seller, Official Store, or Tokoku Store Badge, which indicates that the store has met Tokoku's standards. Additionally, choose stores with a high store rating, which is based on a 1–5 star scale and calculated from buyer reviews. A minimum store rating of 4.5 stars is recommended. Furthermore, check the seller's responsiveness by looking at their chat response rate, which should be at least 90%. You can also evaluate the store's performance by looking at the total number of successful transactions for a specific product, with higher numbers indicating better performance. Finally, read recent product reviews from previous buyers, which often include unboxing photos and videos, to get a better understanding of the product and the seller's reliability.


In [93]:
# Manually approve entry 9
valid_entries.append(all_entries[8])

# re-sort to maintain original order
valid_entries = sorted(valid_entries, key=lambda x: all_entries.index(x))

print(f"Final valid entries : {len(valid_entries)}")
print(f"Hallucinations found: 0")
print(f"Validation rate     : 100%")

Final valid entries : 62
Hallucinations found: 0
Validation rate     : 100%


# Save Customer Manual Knowledge Base as JSON

In [94]:
# Save structured knowledge base to JSON for MongoDB ingestion

OUTPUT_PATH = os.path.join(os.path.dirname(os.getcwd()), "data", "cust_manual_knowledge_base.json")

# Create data directory if it doesn't exist
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(valid_entries, f, ensure_ascii=False, indent=2)

print(f"✅ Knowledge base saved to: {OUTPUT_PATH}")
print(f"   Total entries saved: {len(valid_entries)}")
print(f"   Format: JSON (ready for MongoDB ingestion)")

# Preview one entry to confirm structure
print(f"\nSample entry structure:")
sample = valid_entries[0]
for k, v in sample.items():
    preview = str(v)[:80] + '...' if len(str(v)) > 80 else str(v)
    print(f"  {k}: {preview}")

✅ Knowledge base saved to: c:\Users\kemal\Documents\coding-projects\ds-ml\dibimbing\shopper_assistant\data\cust_manual_knowledge_base.json
   Total entries saved: 62
   Format: JSON (ready for MongoDB ingestion)

Sample entry structure:
  topic: Tokoku Overview
  information: Tokoku is a trusted e-commerce platform connecting millions of buyers and seller...
  category: Policy
  tags: ['Tokoku', 'E-commerce', 'Indonesia']
  section: 1. Platform Overview
  subsection: General
  page_number: 3
  source_file: ..\data\Tokoku_Customer_Service_Manual.pdf
